# Lineage Marker Removal

Train GIN models at depths 0, 2, and 4 on original and marker-reduced inputs, then evaluate:

- **orig/orig:** original training and original validation inputs
- **orig/reduced:** original training and marker-reduced validation inputs
- **reduced/reduced:** marker-reduced training and validation inputs

Validation MSE is summarized per organoid in the full organoid and in crypt, neck, and villus regions. Figure bands show the standard error across organoids. Marker removal is implemented by zeroing channels, so all models retain the same input dimensionality.

**Setup**

In [ ]:
import copy
import json
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "training_data"
sys.path.append(str(PROJECT_ROOT))

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "legend.frameon": False,
})

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())

**Settings**

In [ ]:
# Data and filtering: matched to ablation_analysis.ipynb
EXPERIMENT_GROUP = "lineage_removal"
DATASET_NAME = "fixed_new"
TARGET_INDICES = [0]
TARGET_INDEX_FOR_ANALYSIS = 0

USE_GLOBAL_FEATURES = True
FILTER_BLACKLISTED_ORGANOIDS = True
SUBTRACT_CONSTANT_GLOBAL_BASELINE = False

TIMEPOINT_FILTER_MODE = "rest"  # "rest", "day3p5", or "all"
DAY3P5_TIMEPOINT = "day3p5"
FILL_MISSING_COMPLEXITY = True
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92 # 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Split mode
ANALYSIS_MODE = "folded"  # "single_split" for testing, "folded" for final results
VAL_FRAC = 0.2
N_FOLDS = 5
SPLIT_SEED = 42
FOLDS_TO_RUN = None  # None runs every split; e.g. [0] runs only the first fold

# Models and training
MODEL_DEPTHS = [0, 1, 2, 3, 4]
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 500
PATIENCE = 30
NUM_WORKERS = 4
PREDICT_BATCH_SIZE = 128
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}
SAVE_MODEL_CHECKPOINTS = False

# Region assignment from d_crypts_graph
DCRYPT_FIELD = "d_crypts_graph"
CRYPT_MAX = 0.7
NECK_MAX = 1.20
REGION_ORDER = ["full", "crypt", "neck", "villus"]

# Marker-reduction panels. "all_except_lgr5" is resolved after loading marker names.
REMOVAL_SETS = [
    {"key": "lgr5", "label": "LGR5", "markers": ["LGR5"]},
    {"key": "lyso_agr2", "label": "Lysozyme + Agr2", "markers": ["Lysozyme", "Agr2"]},
    {"key": "sero_chroma", "label": "Serotonin + Chroma", "markers": ["Serotonin", "Chroma"]},
    {
        "key": "lyso_agr2_sero_chroma",
        "label": "Lysozyme + Agr2 +\nSerotonin + Chroma",
        "markers": ["Lysozyme", "Agr2", "Serotonin", "Chroma"],
    },
    {"key": "aldob", "label": "AldoB", "markers": ["AldoB"]},
    {"key": "ki67", "label": "KI67", "markers": ["KI67"]},
    {
        "key": "lgr5_ki67",
        "label": "LGR5 + KI67",
        "markers": ["LGR5", "KI67"],
    },
    {
        "key": "all_except_lgr5",
        "label": "Everything but LGR5",
        "markers": "__ALL_EXCEPT_LGR5__",
    },
]

# Output
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"{ANALYSIS_MODE}_depths-{'-'.join(map(str, MODEL_DEPTHS))}_{RUN_TIMESTAMP}"
SAVE_DIR = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP / RUN_NAME
FIGURES_DIR = SAVE_DIR / "figures"
TABLES_DIR = SAVE_DIR / "tables"
MODELS_DIR = SAVE_DIR / "models"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_MODEL_CHECKPOINTS:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)


def set_all_seeds(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


set_all_seeds(SPLIT_SEED)

**Utilities**

In [ ]:
def save_figure(fig, name, *, dpi=300):
    for suffix in ("pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{name}.{suffix}", dpi=dpi, bbox_inches="tight")


def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, (torch.dtype, np.dtype)):
        return str(obj)
    return obj


def select_target_column(values, target_index=0):
    arr = np.asarray(values)
    if arr.ndim == 1:
        return arr.reshape(-1)
    if arr.ndim == 2 and arr.shape[1] == 1:
        return arr[:, 0]
    if arr.ndim == 2:
        return arr[:, int(target_index)]
    raise ValueError(f"Expected 1-D or 2-D values, got shape {arr.shape}.")


def marker_index(name, marker_names_in):
    names = list(marker_names_in)
    if name in names:
        return names.index(name)
    lower = {str(marker).lower(): i for i, marker in enumerate(names)}
    key = str(name).lower()
    if key not in lower:
        raise ValueError(f"Marker {name!r} not found in {names}.")
    return lower[key]


def resolve_removal_sets(removal_sets, marker_names_in):
    lgr5_idx = marker_index("LGR5", marker_names_in)
    resolved = []
    for spec in removal_sets:
        out = dict(spec)
        if spec["markers"] == "__ALL_EXCEPT_LGR5__":
            out["markers"] = [
                marker for i, marker in enumerate(marker_names_in) if i != lgr5_idx
            ]
        else:
            out["markers"] = list(spec["markers"])
        for marker in out["markers"]:
            marker_index(marker, marker_names_in)
        resolved.append(out)
    return resolved


def zero_marker_channels(graphs_in, markers_to_remove, marker_names_in):
    marker_indices = [
        marker_index(marker, marker_names_in) for marker in markers_to_remove
    ]
    graphs_out = [copy.deepcopy(graph) for graph in graphs_in]
    counts = {marker: 0 for marker in markers_to_remove}
    for graph in graphs_out:
        graph.x = graph.x.clone()
        for marker, marker_idx in zip(markers_to_remove, marker_indices):
            counts[marker] += int((graph.x[:, marker_idx] > 0.5).sum().item())
            graph.x[:, marker_idx] = 0
    return graphs_out, counts


def _nanmin_or_nan(values, axis):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    filled = np.where(finite, arr, np.inf)
    out = np.asarray(np.min(filled, axis=axis), dtype=float)
    out[~np.any(finite, axis=axis)] = np.nan
    return out


def min_node_dcrypt(raw, n_nodes):
    arr = np.asarray(raw, dtype=float)
    if arr.size == 0:
        return np.full(n_nodes, np.nan, dtype=float)
    if arr.ndim == 0:
        return np.full(n_nodes, float(arr), dtype=float)
    if arr.ndim == 1:
        if arr.shape[0] == n_nodes:
            return arr.astype(float)
        if arr.shape[0] == 1:
            return np.full(n_nodes, float(arr[0]), dtype=float)
        raise ValueError(
            f"Cannot align dcrypt length {arr.shape[0]} to {n_nodes} nodes."
        )
    if n_nodes in arr.shape:
        node_axis = list(arr.shape).index(n_nodes)
        moved = np.moveaxis(arr, node_axis, 0).reshape(n_nodes, -1)
        return _nanmin_or_nan(moved, axis=1)
    if arr.ndim >= 2 and arr.shape[1] >= n_nodes:
        values = _nanmin_or_nan(arr, axis=0)
    else:
        values = _nanmin_or_nan(arr, axis=1)
    values = np.asarray(values, dtype=float).reshape(-1)
    if values.shape[0] != n_nodes:
        raise ValueError(
            f"Transformed dcrypt length {values.shape[0]} does not match {n_nodes} nodes."
        )
    return values


def region_masks_for_graph(graph, *, meta_lookup):
    from src.data.metadata import get_graph_metadata

    n_nodes = int(graph.x.shape[0])
    metadata = get_graph_metadata(
        graph, meta_lookup=meta_lookup, strict=False, default={}
    )
    dcrypt = min_node_dcrypt(
        metadata.get(DCRYPT_FIELD, np.asarray([], dtype=float)), n_nodes
    )
    finite = np.isfinite(dcrypt)
    return {
        "full": np.ones(n_nodes, dtype=bool),
        "crypt": finite & (dcrypt < CRYPT_MAX),
        "neck": finite & (dcrypt >= CRYPT_MAX) & (dcrypt <= NECK_MAX),
        "villus": (~finite) | (dcrypt > NECK_MAX),
    }

## 1. Load And Filter Data

In [ ]:
from src.data.filters import (
    filter_graphs_by_blacklist,
    filter_graphs_by_marker_diversity,
    filter_graphs_by_metadata,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
    load_graph_blacklist_from_dir,
)
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    attach_metadata_to_graphs,
    fill_missing_metadata_for_group,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
metadata = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, metadata, exclude_keys=None)
graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    marker_names = [f"marker_{i}" for i in range(int(graphs[0].x.size(1)))]
print(f"Loaded {len(graphs)} organoids and {len(marker_names)} markers.")

if TIMEPOINT_FILTER_MODE == "rest":
    graphs = filter_graphs_by_metadata(
        graphs,
        key="timepoint",
        drop_values={DAY3P5_TIMEPOINT},
        missing="keep",
        inplace=False,
        print_summary=True,
    )
elif TIMEPOINT_FILTER_MODE == "day3p5":
    graphs = filter_graphs_by_metadata(
        graphs,
        key="timepoint",
        keep_values={DAY3P5_TIMEPOINT},
        missing="drop",
        inplace=False,
        print_summary=True,
    )
elif TIMEPOINT_FILTER_MODE not in (None, "all"):
    raise ValueError(
        'TIMEPOINT_FILTER_MODE must be "rest", "day3p5", or "all".'
    )

if FILTER_BLACKLISTED_ORGANOIDS:
    blacklist = load_graph_blacklist_from_dir(data_dir)
    graphs = filter_graphs_by_blacklist(graphs, blacklist, print_summary=True)
else:
    blacklist = set()

if FILL_MISSING_COMPLEXITY:
    graphs = fill_missing_metadata_for_group(
        graphs,
        field="complexity",
        fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
        dataset=MISSING_COMPLEXITY_GROUP["dataset"],
        timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
    )

graphs, spherical = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
)
spherical = filter_graphs_by_marker_diversity(
    spherical,
    min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
    print_summary=True,
)
graphs = filter_graphs_by_numeric_metadata(
    graphs,
    key="complexity",
    min_value=COMPLEXITY_MIN,
    allow_missing=False,
    inplace=False,
    print_summary=True,
)
if INTERPOLATE_TARGET_OUTLIERS:
    graphs, target_outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    target_outlier_info = None

REMOVAL_SETS_RESOLVED = resolve_removal_sets(REMOVAL_SETS, marker_names)
print(f"After filtering: {len(graphs)} organoids.")
display(pd.DataFrame(REMOVAL_SETS_RESOLVED))

## 2. Organoid Splits

In [ ]:
from src.data.splits import graph_metadata_key, train_val_split_graphs


def make_organoid_kfolds(graphs_in, *, n_folds=5, seed=0):
    if len(graphs_in) < n_folds:
        raise ValueError(f"Need at least {n_folds} graphs, got {len(graphs_in)}.")
    keys = [graph_metadata_key(graph) for graph in graphs_in]
    if len(set(keys)) != len(keys):
        raise ValueError("Expected unique organoid metadata keys.")

    rng = np.random.default_rng(seed)
    indices = np.arange(len(graphs_in), dtype=int)
    rng.shuffle(indices)
    val_folds = np.array_split(indices, n_folds)
    splits_out = []
    for fold_id, val_indices in enumerate(val_folds):
        val_indices = np.asarray(val_indices, dtype=int)
        val_set = set(val_indices.tolist())
        train_indices = np.asarray(
            [i for i in range(len(graphs_in)) if i not in val_set],
            dtype=int,
        )
        splits_out.append({
            "fold": fold_id,
            "train_indices": train_indices,
            "val_indices": val_indices,
            "train_keys": [keys[int(i)] for i in train_indices],
            "val_keys": [keys[int(i)] for i in val_indices],
        })
    return splits_out


def make_analysis_splits(graphs_in):
    if ANALYSIS_MODE == "folded":
        return make_organoid_kfolds(
            graphs_in, n_folds=N_FOLDS, seed=SPLIT_SEED
        )
    if ANALYSIS_MODE != "single_split":
        raise ValueError(
            'ANALYSIS_MODE must be "single_split" or "folded".'
        )
    _, _, split_info = train_val_split_graphs(
        graphs_in,
        val_frac=VAL_FRAC,
        seed=SPLIT_SEED,
        key_fn=graph_metadata_key,
        inplace=False,
    )
    return [{
        "fold": 0,
        "train_indices": np.asarray(split_info["train_indices"], dtype=int),
        "val_indices": np.asarray(split_info["val_indices"], dtype=int),
        "train_keys": split_info["train_keys"],
        "val_keys": split_info["val_keys"],
    }]


splits = make_analysis_splits(graphs)
if FOLDS_TO_RUN is not None:
    selected_folds = {int(fold) for fold in FOLDS_TO_RUN}
    splits = [split for split in splits if int(split["fold"]) in selected_folds]
    if not splits:
        raise ValueError("FOLDS_TO_RUN did not select any available folds.")

split_summary_df = pd.DataFrame([{
    "fold": int(split["fold"]),
    "n_train_graphs": len(split["train_indices"]),
    "n_val_graphs": len(split["val_indices"]),
} for split in splits])
split_summary_df.to_csv(TABLES_DIR / "split_summary.csv", index=False)
display(split_summary_df)

## 3. Fold Preparation And Models

In [ ]:
from src.data.metadata import (
    add_log_metadata_features,
    infer_global_dim,
    promote_metadata_to_graph_tensors,
    snapshot_graph_metadata,
    strip_graph_metadata,
)
from src.data.target_transforms import (
    AsinhStandardizeTransform,
    ChainedTargetTransform,
    GlobalBaselineResidualTransform,
    standardize_graph_global_features,
)
from src.models.baseline import GlobalFeatureMLP
from src.models.gnn import GINCurvature
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term

FIELD_SPECS = [{
    "meta_keys": [
        "log_surface_area",
        "log_volume",
        "log_volume_over_area",
        "log_num_cells",
    ],
    "attr_name": "global_feat",
    "kind": "graph_vector",
    "dtype": torch.float32,
}]


def prepare_fold_graphs(train_graphs, val_graphs):
    g_train = copy.deepcopy(train_graphs)
    g_val = copy.deepcopy(val_graphs)

    if USE_GLOBAL_FEATURES:
        g_train = promote_metadata_to_graph_tensors(
            add_log_metadata_features(g_train, inplace=False),
            FIELD_SPECS,
            inplace=False,
        )
        g_val = promote_metadata_to_graph_tensors(
            add_log_metadata_features(g_val, inplace=False),
            FIELD_SPECS,
            inplace=False,
        )

    train_meta_lookup = snapshot_graph_metadata(g_train)
    val_meta_lookup = snapshot_graph_metadata(g_val)
    g_train = strip_graph_metadata(g_train, inplace=False)
    g_val = strip_graph_metadata(g_val, inplace=False)

    if USE_GLOBAL_FEATURES:
        standardize_graph_global_features(
            g_train, g_val, attr_name="global_feat", robust=False
        )

    if SUBTRACT_CONSTANT_GLOBAL_BASELINE:
        target_transform = ChainedTargetTransform([
            GlobalBaselineResidualTransform(num_workers=NUM_WORKERS),
            AsinhStandardizeTransform(robust=True),
        ])
    else:
        target_transform = AsinhStandardizeTransform(robust=True)
    target_transform.fit(g_train)
    target_transform.transform_graphs(g_train, in_place=True)
    target_transform.transform_graphs(g_val, in_place=True)

    return {
        "g_train": g_train,
        "g_val": g_val,
        "train_meta_lookup": train_meta_lookup,
        "val_meta_lookup": val_meta_lookup,
        "target_transform": target_transform,
    }


def infer_target_dim(graphs_in):
    y = graphs_in[0].y
    return 1 if y.ndim == 1 else int(y.shape[1])


def make_gin_model(graphs_in, depth):
    return GINCurvature(
        n_markers=int(graphs_in[0].x.size(1)),
        global_dim=infer_global_dim(graphs_in),
        hidden_dim=HIDDEN_DIM,
        num_layers=int(depth),
        dropout=DROPOUT,
        residual=RESIDUAL,
        norm=NORM,
        target_dim=infer_target_dim(graphs_in),
    )


def make_global_baseline(graphs_in):
    return GlobalFeatureMLP(
        global_dim=infer_global_dim(graphs_in),
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        target_dim=infer_target_dim(graphs_in),
    )


def make_train_config():
    return TrainConfig(
        lr=LR,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        num_workers=NUM_WORKERS,
        aux_losses=[WeightedLossTerm(
            name="edge",
            fn=edge_loss_term,
            weight=EDGE_LOSS_WEIGHT,
            params=EDGE_LOSS_PARAMS,
        )],
    )


cfg = make_train_config()
device = cfg.device
print("device =", device)

## 4. Organoid-Level Regional Evaluation

In [ ]:
from src.inference.predict import predict_targets


def predict_graphwise(graphs_in, model, target_transform):
    y_true, y_pred, _ = predict_targets(
        graphs_in,
        model,
        device=device,
        batch_size=PREDICT_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        target_transform=target_transform,
    )
    y_true = select_target_column(y_true, TARGET_INDEX_FOR_ANALYSIS)
    y_pred = select_target_column(y_pred, TARGET_INDEX_FOR_ANALYSIS)

    output = []
    offset = 0
    for graph in graphs_in:
        n_nodes = int(graph.x.shape[0])
        output.append({
            "y_true": y_true[offset:offset + n_nodes],
            "y_pred": y_pred[offset:offset + n_nodes],
        })
        offset += n_nodes
    if offset != len(y_true):
        raise RuntimeError("Prediction output did not align with graph sizes.")
    return output


def regional_mse_rows(
    graphs_in,
    predictions,
    *,
    meta_lookup,
    fold,
    removal_key,
    removal_label,
    train_eval_key,
    train_eval_label,
    depth=None,
):
    rows = []
    for graph, pred in zip(graphs_in, predictions):
        masks = region_masks_for_graph(graph, meta_lookup=meta_lookup)
        key = graph_metadata_key(graph, meta_lookup=meta_lookup)
        organoid_id = " | ".join(map(str, key))
        squared_error = np.square(pred["y_pred"] - pred["y_true"])
        for region in REGION_ORDER:
            mask = masks[region]
            if not np.any(mask):
                continue
            rows.append({
                "fold": int(fold),
                "organoid_id": organoid_id,
                "removal_key": removal_key,
                "removal_label": removal_label,
                "train_eval_key": train_eval_key,
                "train_eval_label": train_eval_label,
                "depth": np.nan if depth is None else int(depth),
                "region": region,
                "n_nodes": int(mask.sum()),
                "mse": float(np.mean(squared_error[mask])),
            })
    return rows


def summarize_organoid_mse(rows_df, group_columns):
    def summarize(group):
        values = group["mse"].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        n = len(values)
        std = float(np.std(values, ddof=1)) if n > 1 else 0.0
        return pd.Series({
            "mean_mse": float(np.mean(values)) if n else np.nan,
            "std_mse": std if n else np.nan,
            "sem_mse": std / np.sqrt(n) if n else np.nan,
            "n_organoids": int(n),
            "n_nodes": int(group["n_nodes"].sum()),
        })

    return (
        rows_df.groupby(group_columns, dropna=False, sort=False)
        .apply(summarize, include_groups=False)
        .reset_index()
    )

## 5. Train And Evaluate

In folded mode the global-feature baseline is trained once **per fold**, because each fold has a different training set. Within a fold it is reused for every removal panel. Results are written incrementally after each completed model.

In [ ]:
organoid_mse_rows = []
baseline_mse_rows = []
training_rows = []
zero_count_rows = []


def persist_partial_results():
    pd.DataFrame(organoid_mse_rows).to_csv(
        TABLES_DIR / "organoid_level_mse_partial.csv", index=False
    )
    pd.DataFrame(baseline_mse_rows).to_csv(
        TABLES_DIR / "global_baseline_organoid_mse_partial.csv", index=False
    )
    pd.DataFrame(training_rows).to_csv(
        TABLES_DIR / "training_summary_partial.csv", index=False
    )
    pd.DataFrame(zero_count_rows).to_csv(
        TABLES_DIR / "marker_zero_counts_partial.csv", index=False
    )


for split in splits:
    fold = int(split["fold"])
    fold_seed = SPLIT_SEED + 10000 * fold
    print(f"\n{'=' * 72}\nFold {fold + 1}/{len(splits)}\n{'=' * 72}")

    raw_train = [graphs[int(i)] for i in split["train_indices"]]
    raw_val = [graphs[int(i)] for i in split["val_indices"]]
    pack = prepare_fold_graphs(raw_train, raw_val)
    g_train = pack["g_train"]
    g_val = pack["g_val"]
    val_meta_lookup = pack["val_meta_lookup"]
    target_transform = pack["target_transform"]

    # One marker-blind baseline per fold.
    set_all_seeds(fold_seed + 9000)
    print(f"Training global baseline | fold={fold}")
    baseline_model = make_global_baseline(g_train)
    baseline_model, baseline_metrics, baseline_history = train(
        baseline_model, g_train, g_val, cfg
    )
    baseline_predictions = predict_graphwise(
        g_val, baseline_model, target_transform
    )
    baseline_mse_rows.extend(regional_mse_rows(
        g_val,
        baseline_predictions,
        meta_lookup=val_meta_lookup,
        fold=fold,
        removal_key="global_baseline",
        removal_label="Global baseline",
        train_eval_key="global_baseline",
        train_eval_label="Global baseline",
        depth=None,
    ))
    training_rows.append({
        "fold": fold,
        "removal_key": "global_baseline",
        "train_eval_key": "global_baseline",
        "depth": np.nan,
        "val_mae_transformed": float(baseline_metrics["val_mae"]),
        "epochs_trained": len(baseline_history["val_mae"]),
    })
    if SAVE_MODEL_CHECKPOINTS:
        torch.save(
            baseline_model.state_dict(),
            MODELS_DIR / f"fold{fold}_global_baseline.pt",
        )
    del baseline_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Original models are trained once per depth and reused for all eval-only reductions.
    original_models = {}
    for depth in MODEL_DEPTHS:
        set_all_seeds(fold_seed + int(depth))
        print(f"Training original GIN | fold={fold} | depth={depth}")
        model = make_gin_model(g_train, depth)
        model, metrics, history = train(model, g_train, g_val, cfg)
        original_models[int(depth)] = model
        training_rows.append({
            "fold": fold,
            "removal_key": "original",
            "train_eval_key": "orig_orig",
            "depth": int(depth),
            "val_mae_transformed": float(metrics["val_mae"]),
            "epochs_trained": len(history["val_mae"]),
        })
        original_predictions = predict_graphwise(
            g_val, model, target_transform
        )
        for spec in REMOVAL_SETS_RESOLVED:
            organoid_mse_rows.extend(regional_mse_rows(
                g_val,
                original_predictions,
                meta_lookup=val_meta_lookup,
                fold=fold,
                removal_key=spec["key"],
                removal_label=spec["label"],
                train_eval_key="orig_orig",
                train_eval_label="orig/orig",
                depth=depth,
            ))
        if SAVE_MODEL_CHECKPOINTS:
            torch.save(
                model.state_dict(),
                MODELS_DIR / f"fold{fold}_original_depth{depth}.pt",
            )
        persist_partial_results()

    for removal_idx, spec in enumerate(REMOVAL_SETS_RESOLVED):
        removal_key = spec["key"]
        removal_label = spec["label"]
        markers = spec["markers"]
        print(f"\nRemoval: {removal_label} -> {markers}")
        g_train_reduced, train_counts = zero_marker_channels(
            g_train, markers, marker_names
        )
        g_val_reduced, val_counts = zero_marker_channels(
            g_val, markers, marker_names
        )
        for marker in markers:
            zero_count_rows.append({
                "fold": fold,
                "removal_key": removal_key,
                "removal_label": removal_label,
                "marker": marker,
                "train_positive_values_zeroed": int(train_counts[marker]),
                "val_positive_values_zeroed": int(val_counts[marker]),
            })

        for depth in MODEL_DEPTHS:
            depth = int(depth)
            # Original model evaluated on reduced validation inputs.
            eval_only_predictions = predict_graphwise(
                g_val_reduced, original_models[depth], target_transform
            )
            organoid_mse_rows.extend(regional_mse_rows(
                g_val_reduced,
                eval_only_predictions,
                meta_lookup=val_meta_lookup,
                fold=fold,
                removal_key=removal_key,
                removal_label=removal_label,
                train_eval_key="orig_reduced",
                train_eval_label="orig/reduced",
                depth=depth,
            ))

            # Reduced model trained and evaluated with the same zeroed channels.
            model_seed = fold_seed + 100 * (removal_idx + 1) + depth
            set_all_seeds(model_seed)
            print(
                f"Training reduced GIN | fold={fold} | "
                f"removal={removal_key} | depth={depth}"
            )
            reduced_model = make_gin_model(g_train_reduced, depth)
            reduced_model, metrics, history = train(
                reduced_model, g_train_reduced, g_val_reduced, cfg
            )
            training_rows.append({
                "fold": fold,
                "removal_key": removal_key,
                "train_eval_key": "reduced_reduced",
                "depth": depth,
                "val_mae_transformed": float(metrics["val_mae"]),
                "epochs_trained": len(history["val_mae"]),
            })
            reduced_predictions = predict_graphwise(
                g_val_reduced, reduced_model, target_transform
            )
            organoid_mse_rows.extend(regional_mse_rows(
                g_val_reduced,
                reduced_predictions,
                meta_lookup=val_meta_lookup,
                fold=fold,
                removal_key=removal_key,
                removal_label=removal_label,
                train_eval_key="reduced_reduced",
                train_eval_label="reduced/reduced",
                depth=depth,
            ))
            if SAVE_MODEL_CHECKPOINTS:
                torch.save(
                    reduced_model.state_dict(),
                    MODELS_DIR / (
                        f"fold{fold}_{removal_key}_depth{depth}.pt"
                    ),
                )
            del reduced_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            persist_partial_results()

        del g_train_reduced, g_val_reduced

    del original_models
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Finished all requested folds.")

## 6. Summaries

In [ ]:
organoid_mse_df = pd.DataFrame(organoid_mse_rows)
baseline_mse_df = pd.DataFrame(baseline_mse_rows)
training_summary_df = pd.DataFrame(training_rows)
zero_counts_df = pd.DataFrame(zero_count_rows)

required_result_columns = {
    "fold", "organoid_id", "removal_key", "train_eval_key",
    "depth", "region", "mse",
}
missing = required_result_columns - set(organoid_mse_df.columns)
if missing:
    raise RuntimeError(f"Missing result columns: {sorted(missing)}")

mse_summary_df = summarize_organoid_mse(
    organoid_mse_df,
    [
        "removal_key",
        "removal_label",
        "train_eval_key",
        "train_eval_label",
        "depth",
        "region",
    ],
)
baseline_summary_df = summarize_organoid_mse(
    baseline_mse_df,
    ["region"],
)

organoid_mse_df.to_csv(
    TABLES_DIR / "organoid_level_mse.csv", index=False
)
mse_summary_df.to_csv(
    TABLES_DIR / "mse_summary_across_organoids.csv", index=False
)
baseline_mse_df.to_csv(
    TABLES_DIR / "global_baseline_organoid_mse.csv", index=False
)
baseline_summary_df.to_csv(
    TABLES_DIR / "global_baseline_summary_across_organoids.csv", index=False
)
training_summary_df.to_csv(
    TABLES_DIR / "training_summary.csv", index=False
)
zero_counts_df.to_csv(
    TABLES_DIR / "marker_zero_counts.csv", index=False
)

display(mse_summary_df.head(12))
display(baseline_summary_df)

## 7. Regional MSE Figures

In [ ]:
COMBO_ORDER = ["orig_orig", "orig_reduced", "reduced_reduced"]
COMBO_STYLE = {
    "orig_orig": {
        "label": "orig/orig",
        "color": "#2f6f9f",
        "marker": "o",
        "linestyle": "-",
    },
    "orig_reduced": {
        "label": "orig/reduced",
        "color": "#c65f32",
        "marker": "s",
        "linestyle": "--",
    },
    "reduced_reduced": {
        "label": "reduced/reduced",
        "color": "#3f8664",
        "marker": "^",
        "linestyle": "-.",
    },
}


def plot_region_mse(region):
    region_df = mse_summary_df[mse_summary_df["region"] == region]
    baseline_row = baseline_summary_df[
        baseline_summary_df["region"] == region
    ]
    if baseline_row.empty:
        raise ValueError(f"No global baseline result for region {region!r}.")
    baseline_mean = float(baseline_row.iloc[0]["mean_mse"])
    baseline_sem = float(baseline_row.iloc[0]["sem_mse"])

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(15.5, 7.2),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    axes = axes.ravel()
    legend_handles = []
    legend_labels = []

    for panel_idx, (ax, spec) in enumerate(zip(axes, REMOVAL_SETS_RESOLVED)):
        panel = region_df[region_df["removal_key"] == spec["key"]]
        for combo in COMBO_ORDER:
            style = COMBO_STYLE[combo]
            curve = (
                panel[panel["train_eval_key"] == combo]
                .sort_values("depth")
            )
            x = curve["depth"].to_numpy(dtype=float)
            y = curve["mean_mse"].to_numpy(dtype=float)
            sem = curve["sem_mse"].to_numpy(dtype=float)
            line, = ax.plot(
                x,
                y,
                color=style["color"],
                marker=style["marker"],
                linestyle=style["linestyle"],
                linewidth=2.0,
                markersize=5,
                label=style["label"],
            )
            ax.fill_between(
                x,
                y - sem,
                y + sem,
                color=style["color"],
                alpha=0.18,
                linewidth=0,
            )
            if panel_idx == 0:
                legend_handles.append(line)
                legend_labels.append(style["label"])

        baseline_line = ax.axhline(
            baseline_mean,
            color="#4a4a4a",
            linestyle=":",
            linewidth=1.8,
            label="Global baseline",
        )
        ax.axhspan(
            baseline_mean - baseline_sem,
            baseline_mean + baseline_sem,
            color="#6b6b6b",
            alpha=0.14,
            linewidth=0,
        )
        ax.set_title(spec["label"], fontsize=10)
        ax.set_xticks(MODEL_DEPTHS)
        ax.set_xlabel("Model depth")
        ax.set_ylabel("MSE")
        ax.grid(alpha=0.25)

    legend_handles.append(baseline_line)
    legend_labels.append("Global baseline")
    fig.legend(
        legend_handles,
        legend_labels,
        loc="outside lower center",
        ncol=4,
        bbox_to_anchor=(0.5, -0.035),
    )
    fig.suptitle(
        f"Lineage removal: {region.capitalize()} region",
        fontsize=15,
        fontweight="bold",
    )
    save_figure(fig, f"lineage_removal_mse_{region}")
    return fig


region_figures = {
    region: plot_region_mse(region) for region in REGION_ORDER
}
plt.show()

## 8. Save Configuration

In [ ]:
config = {
    "created_at": RUN_TIMESTAMP,
    "project_root": PROJECT_ROOT,
    "save_dir": SAVE_DIR,
    "dataset": {
        "name": DATASET_NAME,
        "target_indices": TARGET_INDICES,
        "target_index_for_analysis": TARGET_INDEX_FOR_ANALYSIS,
        "marker_names": marker_names,
    },
    "filtering": {
        "timepoint_filter_mode": TIMEPOINT_FILTER_MODE,
        "day3p5_timepoint": DAY3P5_TIMEPOINT,
        "filter_blacklisted_organoids": FILTER_BLACKLISTED_ORGANOIDS,
        "fill_missing_complexity": FILL_MISSING_COMPLEXITY,
        "missing_complexity_group": MISSING_COMPLEXITY_GROUP,
        "sphericity_max": SPHERICITY_MAX,
        "spherical_marker_diversity_min": SPHERICAL_MARKER_DIVERSITY_MIN,
        "complexity_min": COMPLEXITY_MIN,
        "interpolate_target_outliers": INTERPOLATE_TARGET_OUTLIERS,
        "outlier_clip_quantiles": OUTLIER_CLIP_QUANTILES,
        "n_graphs_after_filtering": len(graphs),
    },
    "split": {
        "analysis_mode": ANALYSIS_MODE,
        "val_frac": VAL_FRAC,
        "n_folds": N_FOLDS,
        "split_seed": SPLIT_SEED,
        "folds_to_run": FOLDS_TO_RUN,
        "folds_completed": sorted(organoid_mse_df["fold"].unique().tolist()),
    },
    "model": {
        "class": "GINCurvature",
        "depths": MODEL_DEPTHS,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "norm": NORM,
        "residual": RESIDUAL,
        "use_global_features": USE_GLOBAL_FEATURES,
        "subtract_constant_global_baseline": SUBTRACT_CONSTANT_GLOBAL_BASELINE,
    },
    "training": {
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "predict_batch_size": PREDICT_BATCH_SIZE,
        "edge_loss_weight": EDGE_LOSS_WEIGHT,
        "edge_loss_params": EDGE_LOSS_PARAMS,
        "save_model_checkpoints": SAVE_MODEL_CHECKPOINTS,
    },
    "regions": {
        "dcrypt_field": DCRYPT_FIELD,
        "crypt": f"d < {CRYPT_MAX}",
        "neck": f"{CRYPT_MAX} <= d <= {NECK_MAX}",
        "villus": f"d > {NECK_MAX} or non-finite",
    },
    "removal_sets": REMOVAL_SETS_RESOLVED,
    "evaluation": {
        "combinations": {
            "orig_orig": "original train / original validation",
            "orig_reduced": "original train / reduced validation",
            "reduced_reduced": "reduced train / reduced validation",
        },
        "aggregation_unit": "organoid",
        "uncertainty": "standard error across organoid-level MSE values",
        "baseline": "one GlobalFeatureMLP per fold, reused across removal panels",
    },
}

with open(SAVE_DIR / "settings.json", "w") as handle:
    json.dump(jsonable(config), handle, indent=2)

print(f"Saved figures, tables, and settings to {SAVE_DIR}")